In [ ]:
import pandas as pd
import torch
import numpy as np
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

# ---------------------------------------------
# 0. 메모리 초기화
# ---------------------------------------------
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------
# 1. 데이터 로드 및 전처리
# ---------------------------------------------
train_combined_path = '../Xuxeong/final_train_data.csv'
df = pd.read_csv(train_combined_path)
df = df.dropna(subset=['conversation', 'label'])

# Train / Validation 분리 (8:2)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# ---------------------------------------------
# 2. 모델 및 토크나이저 로드 (하이퍼파라미터 원본 유지)
# ---------------------------------------------
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 5          
LEARNING_RATE = 1e-5

MODEL_NAME = "beomi/KcELECTRA-base-v2022" 
num_labels = len(df['label'].unique()) 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# ---------------------------------------------
# 3. Custom Dataset 클래스
# ---------------------------------------------
class KcElectraDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = KcElectraDataset(train_df['conversation'].values, train_df['label'].values, tokenizer, MAX_LENGTH)
val_dataset = KcElectraDataset(val_df['conversation'].values, val_df['label'].values, tokenizer, MAX_LENGTH)

# ---------------------------------------------
# 4. 평가지표 계산 함수
# ---------------------------------------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    
    return {
        'accuracy': acc,
        'f1_macro': f1
    }

# ---------------------------------------------
# 5. Trainer 셋팅 및 학습 (Training)
# ---------------------------------------------
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,                    # EPOCHS = 5
    per_device_train_batch_size=32,        # BATCH_SIZE = 32
    per_device_eval_batch_size=32,
    learning_rate=1e-5,                    # LEARNING_RATE = 1e-5
    weight_decay=0.01,                     # Weight Decay = 0.01
    adam_epsilon=1e-5,                     # Epsilon = 1e-5
    max_grad_norm=1.0,                     # Max Grad Norm = 1.0
    warmup_ratio=0.1,                      # SCHEDULER = 10% Warmup
    lr_scheduler_type="linear",            # SCHEDULER = Linear
    seed=42,                               # SEED = 42
    eval_strategy="epoch",                 # (에러 수정됨) 에포크마다 평가
    save_strategy="epoch",                 
    logging_steps=50,                      
    report_to="none"                       
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("▶ [1/2] 모델 학습을 시작합니다...\n")
trainer.train()

# ==========================================
# 6. 최종 평가 및 리포트 출력 (Evaluation)
# ==========================================
print("\n▶ [2/2] 최종 모델 성능 평가를 진행합니다...")

# 기본 metrics 확인 (eval_loss, eval_accuracy 등)
eval_results = trainer.evaluate()

print("-" * 50)
print("🎯 [최종 검증 세트(Validation Set) 평가 결과]")
print(f" - Loss (손실): {eval_results['eval_loss']:.4f}")
print(f" - Accuracy (정확도): {eval_results['eval_accuracy'] * 100:.2f}%")
print(f" - F1 Score (Macro): {eval_results['eval_f1_macro']:.4f}")
print("-" * 50)

print("\n▶ 검증 데이터셋 상세 예측 중...")
# val_dataset에 대해 예측을 수행하여 로짓(logits) 추출
output = trainer.predict(val_dataset)
preds = np.argmax(output.predictions, axis=-1)

# 타겟 이름 맵핑 (0~4번 인덱스 순서에 맞게 설정)
target_names = ['협박 대화', '갈취 대화', '직장 내 괴롭힘 대화', '기타 괴롭힘 대화', '일반 대화']

print("\n" + "="*60)
print("🎯 [KcELECTRA 상세 성능 리포트]")
print("="*60)
# 실제 정답(val_df['label'])과 모델의 예측값(preds) 비교
print(classification_report(val_df['label'].values, preds, target_names=target_names, zero_division=0))
print("="*60)

# 학습된 모델 및 토크나이저 안전하게 저장
save_directory = "./best_kcelectra_model2"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"\n✅ 학습 완료! 모델과 토크나이저가 '{save_directory}' 폴더에 안전하게 저장되었습니다.")